# Skubal Next-Pitch Type Prediction

Multi-class classification of the next pitch Tarik Skubal (MLBAM 669373) will throw, given pre-pitch context only. Five effective classes: FF, CH, SL, SI, CU.

Pipeline:
1. Load all season parquets, filter to Skubal
2. Build pre-pitch features (count, base state, handedness, score diff, inning, lag-1 / lag-2 prior pitch types within at-bat)
3. Time-based split: train 2020-2024, val 2025 first half, test 2025 second half + 2026
4. Baseline: always-FF and Logistic Regression and Random Forest
5. Improved: XGBoost with sequence features, batter ID, hyperparameter search
6. Persist best model and metrics


In [1]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    log_loss,
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT = Path('.')
DATA = PROJECT / 'data'
DELIV = PROJECT / 'deliverables'
DELIV.mkdir(parents=True, exist_ok=True)
PITCHER_ID = 669373
TARGET_CLASSES = ['CH', 'CU', 'FF', 'SI', 'SL']  # alphabetical, the 5 effective classes
print('Setup OK')


Setup OK


## 1. Load Skubal data

In [2]:
frames = []
keep_cols = [
    'game_pk', 'game_date', 'game_year', 'at_bat_number', 'pitch_number',
    'pitcher', 'batter', 'pitch_type', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'stand', 'p_throws',
    'on_1b', 'on_2b', 'on_3b',
    'bat_score', 'fld_score', 'home_score', 'away_score',
]
for year in range(2020, 2027):
    fp = DATA / f'statcast_{year}.parquet'
    if not fp.exists():
        continue
    schema_cols = pq.read_schema(fp).names
    cols = [c for c in keep_cols if c in schema_cols]
    df = pd.read_parquet(fp, columns=cols)
    df = df[df['pitcher'] == PITCHER_ID].copy()
    if not df.empty:
        frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw['game_date'] = pd.to_datetime(raw['game_date'])
print('Raw rows:', len(raw))
print('Years:', sorted(raw['game_year'].unique()))
print('Pitch type counts:')
print(raw['pitch_type'].value_counts(dropna=False))


Raw rows: 13935
Years: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
Pitch type counts:
pitch_type
FF      4906
CH      3058
SL      2573
SI      2462
CU       697
None     120
FS        97
FC        19
KC         3
Name: count, dtype: int64


## 2. Build pre-pitch features

Target = current `pitch_type` (the pitch about to be thrown, predicted from state available before release).
Lag features = previous pitches in the same at-bat. Features describe state at the moment of the decision.

In [3]:
df = raw.sort_values(['game_date', 'game_pk', 'at_bat_number', 'pitch_number']).reset_index(drop=True)

# Filter to 5 effective classes
df = df[df['pitch_type'].isin(TARGET_CLASSES)].copy()

# Base-state flags (on_1b is batter-id-or-NA in Statcast)
for c in ['on_1b', 'on_2b', 'on_3b']:
    df[c + '_flag'] = df[c].notna().astype(int)

# Score differential from pitcher's perspective
df['score_diff'] = df['fld_score'].fillna(0) - df['bat_score'].fillna(0)

# Categorical encodes
df['stand_R'] = (df['stand'] == 'R').astype(int)

# Lag pitch types within at-bat
grp = df.groupby(['game_pk', 'at_bat_number'], sort=False)
df['prev_pitch_1'] = grp['pitch_type'].shift(1).fillna('NONE')
df['prev_pitch_2'] = grp['pitch_type'].shift(2).fillna('NONE')
df['prev_pitch_3'] = grp['pitch_type'].shift(3).fillna('NONE')
df['ab_pitch_idx'] = grp.cumcount()  # position within the at-bat (0 = first pitch)

# One-hot the lag pitch type strings (incl. NONE for first pitch of the at-bat)
lag_levels = TARGET_CLASSES + ['NONE']
for lag in ['prev_pitch_1', 'prev_pitch_2', 'prev_pitch_3']:
    for lvl in lag_levels:
        df[f'{lag}_{lvl}'] = (df[lag] == lvl).astype(int)

print('Modeling rows:', len(df))
print('Per year:', df.groupby('game_year').size().to_dict())
print('Class distribution:')
print(df['pitch_type'].value_counts(normalize=True).round(3))


Modeling rows: 13696
Per year: {2020: 584, 2021: 2673, 2022: 2141, 2023: 1215, 2024: 3258, 2025: 3217, 2026: 608}
Class distribution:
pitch_type
FF    0.358
CH    0.223
SL    0.188
SI    0.180
CU    0.051
Name: proportion, dtype: float64


In [4]:
# Feature matrix
numeric_feats = [
    'balls', 'strikes', 'outs_when_up', 'inning',
    'on_1b_flag', 'on_2b_flag', 'on_3b_flag',
    'score_diff', 'stand_R', 'ab_pitch_idx',
]
lag_feats = [c for c in df.columns if c.startswith(('prev_pitch_1_', 'prev_pitch_2_', 'prev_pitch_3_'))]
FEATS = numeric_feats + lag_feats
FEATS_BASIC = numeric_feats + [c for c in lag_feats if c.startswith('prev_pitch_1_')]

df = df.dropna(subset=numeric_feats + ['pitch_type']).copy()
print('Final rows:', len(df), 'Total feats:', len(FEATS), 'Basic feats:', len(FEATS_BASIC))


Final rows: 13696 Total feats: 28 Basic feats: 16


## 3. Time-based split

- train: 2020-2024
- val: 2025 first half (game_date < 2025-07-01)
- test: 2025 second half + 2026


In [5]:
train_mask = df['game_year'] <= 2024
val_mask = (df['game_year'] == 2025) & (df['game_date'] < '2025-07-01')
test_mask = ((df['game_year'] == 2025) & (df['game_date'] >= '2025-07-01')) | (df['game_year'] == 2026)

X_train, y_train = df.loc[train_mask, FEATS], df.loc[train_mask, 'pitch_type']
X_val, y_val = df.loc[val_mask, FEATS], df.loc[val_mask, 'pitch_type']
X_test, y_test = df.loc[test_mask, FEATS], df.loc[test_mask, 'pitch_type']

X_train_b = df.loc[train_mask, FEATS_BASIC]
X_val_b = df.loc[val_mask, FEATS_BASIC]
X_test_b = df.loc[test_mask, FEATS_BASIC]

print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)
print('Train class distribution:')
print(y_train.value_counts(normalize=True).round(3))


Train: (9871, 28) Val: (1625, 28) Test: (2200, 28)
Train class distribution:
pitch_type
FF    0.375
SL    0.210
CH    0.194
SI    0.161
CU    0.060
Name: proportion, dtype: float64


## 4. Baselines

In [6]:
def evaluate(name, y_true, y_pred, y_proba=None, classes=None):
    out = {
        'model': name,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }
    if y_proba is not None and classes is not None:
        try:
            out['log_loss'] = float(log_loss(y_true, y_proba, labels=classes))
        except Exception:
            out['log_loss'] = None
    rep = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    out['per_class_f1'] = {k: rep[k]['f1-score'] for k in TARGET_CLASSES if k in rep}
    return out

# Majority-class baseline (always FF)
y_pred_ff = np.array(['FF'] * len(y_test))
majority_metrics = evaluate('majority_FF', y_test, y_pred_ff)
print('Majority FF baseline (test):')
print(json.dumps(majority_metrics, indent=2))


Majority FF baseline (test):
{
  "model": "majority_FF",
  "accuracy": 0.3468181818181818,
  "macro_f1": 0.10300371245359433,
  "weighted_f1": 0.17861780136839195,
  "per_class_f1": {
    "CH": 0.0,
    "CU": 0.0,
    "FF": 0.5150185622679716,
    "SI": 0.0,
    "SL": 0.0
  }
}


In [7]:
# Logistic Regression baseline (basic features, lag-1 only)
scaler = StandardScaler()
X_train_b_s = scaler.fit_transform(X_train_b)
X_val_b_s = scaler.transform(X_val_b)
X_test_b_s = scaler.transform(X_test_b)

logreg = LogisticRegression(max_iter=2000, class_weight='balanced')
logreg.fit(X_train_b_s, y_train)
y_pred_lr = logreg.predict(X_test_b_s)
y_proba_lr = logreg.predict_proba(X_test_b_s)
logreg_metrics = evaluate('logreg_basic', y_test, y_pred_lr, y_proba_lr, list(logreg.classes_))
print('LogReg metrics:', json.dumps(logreg_metrics, indent=2))


LogReg metrics: {
  "model": "logreg_basic",
  "accuracy": 0.2840909090909091,
  "macro_f1": 0.25520315283882844,
  "weighted_f1": 0.30248197084362477,
  "log_loss": 1.5802587802953951,
  "per_class_f1": {
    "CH": 0.41352201257861637,
    "CU": 0.11572327044025157,
    "FF": 0.2446043165467626,
    "SI": 0.3856613102595797,
    "SL": 0.11650485436893204
  }
}


In [8]:
# Random Forest baseline (basic features)
rf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=20,
    class_weight='balanced', n_jobs=-1, random_state=42,
)
rf.fit(X_train_b, y_train)
y_pred_rf = rf.predict(X_test_b)
y_proba_rf = rf.predict_proba(X_test_b)
rf_metrics = evaluate('rf_basic', y_test, y_pred_rf, y_proba_rf, list(rf.classes_))
rf_cm = confusion_matrix(y_test, y_pred_rf, labels=TARGET_CLASSES).tolist()
print('Random Forest metrics:')
print(json.dumps(rf_metrics, indent=2))
print('Confusion matrix (rows=true, cols=pred,', TARGET_CLASSES, '):')
for row in rf_cm:
    print(row)

# Top features (basic RF)
rf_imp = pd.Series(rf.feature_importances_, index=X_train_b.columns).sort_values(ascending=False)
print('Top 10 features (RF basic):')
print(rf_imp.head(10))


Random Forest metrics:
{
  "model": "rf_basic",
  "accuracy": 0.2963636363636364,
  "macro_f1": 0.26185978492505885,
  "weighted_f1": 0.3129092318917877,
  "log_loss": 1.5365968290312293,
  "per_class_f1": {
    "CH": 0.40423921271763813,
    "CU": 0.09711684370257967,
    "FF": 0.2648888888888889,
    "SI": 0.40606767794632437,
    "SL": 0.136986301369863
  }
}
Confusion matrix (rows=true, cols=pred, ['CH', 'CU', 'FF', 'SI', 'SL'] ):
[267, 164, 95, 76, 47]
[20, 32, 10, 3, 2]
[247, 206, 149, 118, 43]
[84, 86, 61, 174, 36]
[54, 104, 47, 45, 30]


Top 10 features (RF basic):
stand_R            0.197845
inning             0.113076
balls              0.103976
prev_pitch_1_CH    0.088296
score_diff         0.085347
ab_pitch_idx       0.084266
strikes            0.067535
prev_pitch_1_FF    0.060223
outs_when_up       0.054083
on_1b_flag         0.042685
dtype: float64


## 5. Improved XGBoost

Adds lag-2 and lag-3 prior pitch one-hots, batter id (categorical), hyperparameter search on the validation set.

In [9]:
# Add batter id as a frequency-encoded feature (avoids 1000+ one-hot columns)
batter_freq = df.loc[train_mask, 'batter'].value_counts(normalize=True)
df['batter_freq'] = df['batter'].map(batter_freq).fillna(0.0)

FEATS_PLUS = FEATS + ['batter_freq']
X_train_p = df.loc[train_mask, FEATS_PLUS]
X_val_p = df.loc[val_mask, FEATS_PLUS]
X_test_p = df.loc[test_mask, FEATS_PLUS]

# Encode labels for XGB
label_map = {c: i for i, c in enumerate(TARGET_CLASSES)}
inv_label = {i: c for c, i in label_map.items()}
y_train_e = y_train.map(label_map).astype(int).values
y_val_e = y_val.map(label_map).astype(int).values
y_test_e = y_test.map(label_map).astype(int).values
print('FEATS_PLUS size:', len(FEATS_PLUS))


FEATS_PLUS size: 29


In [10]:
# Compact hyperparam search on validation set
param_grid = [
    {'max_depth': 4, 'learning_rate': 0.1, 'n_estimators': 200, 'min_child_weight': 5, 'subsample': 0.9, 'colsample_bytree': 0.8},
    {'max_depth': 6, 'learning_rate': 0.07, 'n_estimators': 250, 'min_child_weight': 3, 'subsample': 0.9, 'colsample_bytree': 0.8},
    {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.7},
    {'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 250, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.7},
]

best = None
results = []
for i, p in enumerate(param_grid):
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=len(TARGET_CLASSES),
        tree_method='hist', n_jobs=-1, random_state=42,
        eval_metric='mlogloss', **p,
    )
    model.fit(X_train_p, y_train_e)
    val_pred = model.predict(X_val_p)
    val_acc = accuracy_score(y_val_e, val_pred)
    val_f1 = f1_score(y_val_e, val_pred, average='macro', zero_division=0)
    results.append({'idx': i, 'params': p, 'val_acc': float(val_acc), 'val_macro_f1': float(val_f1)})
    print(f'Cfg {i}: acc={val_acc:.4f}  macroF1={val_f1:.4f}  params={p}')
    if best is None or val_f1 > best['val_macro_f1']:
        best = {'idx': i, 'params': p, 'val_acc': float(val_acc), 'val_macro_f1': float(val_f1)}

print()
print('Best config idx:', best['idx'])
print('Best params:', best['params'])


Cfg 0: acc=0.3440  macroF1=0.2413  params={'max_depth': 4, 'learning_rate': 0.1, 'n_estimators': 200, 'min_child_weight': 5, 'subsample': 0.9, 'colsample_bytree': 0.8}


Cfg 1: acc=0.3286  macroF1=0.2484  params={'max_depth': 6, 'learning_rate': 0.07, 'n_estimators': 250, 'min_child_weight': 3, 'subsample': 0.9, 'colsample_bytree': 0.8}


Cfg 2: acc=0.3292  macroF1=0.2433  params={'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 300, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.7}


Cfg 3: acc=0.3089  macroF1=0.2281  params={'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 250, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.7}

Best config idx: 1
Best params: {'max_depth': 6, 'learning_rate': 0.07, 'n_estimators': 250, 'min_child_weight': 3, 'subsample': 0.9, 'colsample_bytree': 0.8}


In [11]:
# Refit best on train+val for final test eval
X_tv = pd.concat([X_train_p, X_val_p], ignore_index=True)
y_tv = np.concatenate([y_train_e, y_val_e])

final_xgb = xgb.XGBClassifier(
    objective='multi:softprob', num_class=len(TARGET_CLASSES),
    tree_method='hist', n_jobs=-1, random_state=42,
    eval_metric='mlogloss', **best['params'],
)
final_xgb.fit(X_tv, y_tv, verbose=False)

y_pred_xgb_e = final_xgb.predict(X_test_p)
y_proba_xgb = final_xgb.predict_proba(X_test_p)
y_pred_xgb = np.array([inv_label[i] for i in y_pred_xgb_e])

xgb_metrics = evaluate('xgb_full', y_test, y_pred_xgb, y_proba_xgb, TARGET_CLASSES)
xgb_cm = confusion_matrix(y_test, y_pred_xgb, labels=TARGET_CLASSES).tolist()
print('XGB final metrics:')
print(json.dumps(xgb_metrics, indent=2))
print('Confusion matrix (rows=true, cols=pred,', TARGET_CLASSES, '):')
for row in xgb_cm:
    print(row)

xgb_imp = pd.Series(final_xgb.feature_importances_, index=FEATS_PLUS).sort_values(ascending=False)
print('Top 15 features (XGB final):')
print(xgb_imp.head(15))


XGB final metrics:
{
  "model": "xgb_full",
  "accuracy": 0.38727272727272727,
  "macro_f1": 0.2713604715430423,
  "weighted_f1": 0.3679076318483523,
  "log_loss": 1.4132599550125948,
  "per_class_f1": {
    "CH": 0.39937353171495693,
    "CU": 0.02666666666666667,
    "FF": 0.48021978021978023,
    "SI": 0.3469387755102041,
    "SL": 0.1036036036036036
  }
}
Confusion matrix (rows=true, cols=pred, ['CH', 'CU', 'FF', 'SI', 'SL'] ):
[255, 2, 293, 64, 35]
[22, 1, 27, 10, 7]
[191, 2, 437, 88, 45]
[89, 2, 160, 136, 54]
[71, 1, 140, 45, 23]
Top 15 features (XGB final):
stand_R              0.162261
prev_pitch_1_CH      0.062228
strikes              0.039777
prev_pitch_1_FF      0.039234
balls                0.036079
prev_pitch_1_NONE    0.035903
prev_pitch_1_SI      0.033978
prev_pitch_3_SI      0.033125
prev_pitch_2_FF      0.032419
prev_pitch_2_CH      0.032292
prev_pitch_3_NONE    0.029872
prev_pitch_2_SI      0.029771
prev_pitch_1_SL      0.029261
prev_pitch_1_CU      0.028870
on_1b_fla

## 6. Persist artifacts

In [12]:
model_payload = {
    'model': final_xgb,
    'feature_order': FEATS_PLUS,
    'classes': TARGET_CLASSES,
    'label_map': label_map,
    'best_params': best['params'],
    'batter_freq_lookup': batter_freq.to_dict(),
}
joblib.dump(model_payload, DELIV / 'skubal_xgb.pkl')

metrics = {
    'pitcher_id': PITCHER_ID,
    'classes': TARGET_CLASSES,
    'rows': {
        'train': int(train_mask.sum()),
        'val': int(val_mask.sum()),
        'test': int(test_mask.sum()),
    },
    'baseline_majority_FF': majority_metrics,
    'logreg_basic': logreg_metrics,
    'rf_basic': rf_metrics,
    'rf_confusion_matrix': {'order': TARGET_CLASSES, 'matrix': rf_cm},
    'xgb_full': xgb_metrics,
    'xgb_confusion_matrix': {'order': TARGET_CLASSES, 'matrix': xgb_cm},
    'xgb_search_results': results,
    'xgb_top_features': xgb_imp.head(20).to_dict(),
    'rf_top_features': rf_imp.head(20).to_dict(),
}
with open(DELIV / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=float)

print('Saved:', DELIV / 'skubal_xgb.pkl')
print('Saved:', DELIV / 'metrics.json')
print()
print('SUMMARY (test set):')
print(f"  majority_FF  : acc={majority_metrics['accuracy']:.4f}  macroF1={majority_metrics['macro_f1']:.4f}")
print(f"  logreg_basic : acc={logreg_metrics['accuracy']:.4f}  macroF1={logreg_metrics['macro_f1']:.4f}")
print(f"  rf_basic     : acc={rf_metrics['accuracy']:.4f}  macroF1={rf_metrics['macro_f1']:.4f}")
print(f"  xgb_full     : acc={xgb_metrics['accuracy']:.4f}  macroF1={xgb_metrics['macro_f1']:.4f}")


Saved: deliverables//skubal_xgb.pkl
Saved: deliverables//metrics.json

SUMMARY (test set):
  majority_FF  : acc=0.3468  macroF1=0.1030
  logreg_basic : acc=0.2841  macroF1=0.2552
  rf_basic     : acc=0.2964  macroF1=0.2619
  xgb_full     : acc=0.3873  macroF1=0.2714
